In [4]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

ModuleNotFoundError: No module named 'langchain'

In [ ]:
loader = PyPDFLoader("2nd semester marks carks.pdf")
documents = loader.load()

In [ ]:
print(documents[0].page_content[:500])

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))

In [ ]:
print(chunks[0].page_content)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
query = "When is my SGPA?"
docs = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(docs):
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print(doc.page_content)

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto"
)

In [ ]:
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0,
    do_sample=False,
    repetition_penalty=1.1
)


In [ ]:
llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    return_source_documents=True
)

In [ ]:
question = "When is my SGPA?"

result = qa_chain(question)

print("ANSWER:\n", result["result"])

In [ ]:
print("\nSOURCE DOCUMENTS:\n")
for doc in result["source_documents"]:
    print(doc.page_content)
    print("----")

In [ ]:
qa_chain("What is my bank balance?")